In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

In [ ]:
folder_path = r"/storage/alplakes_test/lucerne_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_lvl0")

output_folder = os.path.join(folder_path, "outputs_swirl", "ke_eddy")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
lvl0_csv_path = os.path.join(input_folder, "lvl0_20250101_20251231_concat_corr.csv")
lake_csv_path = os.path.join(input_folder, "lake_characteristics_20250101_20251231_concat_corr.csv")

In [ ]:
df_lvl0 = pd.read_csv(lvl0_csv_path)
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])

In [ ]:
df_lake = pd.read_csv(lake_csv_path)
df_lake = df_lake.set_index('id', drop=False)
df_lake['date'] = pd.to_datetime(df_lake['date'])

# Entire lake analysis

In [ ]:
eddy_ke = df_lvl0.groupby('date').sum()['kinetic_energy_eddy_[MJ]']

In [ ]:
lake_ke = df_lake.groupby('date').sum()['kinetic_energy_[MJ]']

In [ ]:
eddy_ke.plot()

In [ ]:
eddy_ke.reset_index().to_csv(os.path.join(output_folder, "ke_eddies.csv"))
lake_ke.reset_index().to_csv(os.path.join(output_folder, "ke_lake.csv"))

eddy_ke = pd.read_csv(os.path.join(output_folder, "ke_eddies.csv"), index_col=0)
lake_ke = pd.read_csv(os.path.join(output_folder, "ke_lake.csv"), index_col=0)
eddy_ke = eddy_ke.set_index('date')
lake_ke = lake_ke.set_index('date')

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
lake_ke.plot(ax=ax)
eddy_ke.plot(ax=ax)
plt.ylabel('Kinetic Energy [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
ax.legend(['Entire lake', 'Eddies'])
#ax.set_yscale('log')
fig.savefig(os.path.join(output_folder, "kinetic_energy.png"))

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax1 = fig.add_subplot(1, 1, 1)

# Left axis: lake KE
lake_ke.plot(ax=ax1)
ax1.set_ylabel('Lake Kinetic Energy [MJ]')

# Right axis: eddy KE
ax2 = ax1.twinx()
eddy_ke.plot(ax=ax2, color='darkorange')
ax2.set_ylabel('Eddy Kinetic Energy [MJ]')

# Shared x-axis formatting
ax1.set_xlabel('')
plt.xticks(rotation=10)

# Legend (manual, since two axes)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, ['Entire lake', 'Eddies'])

ax1.grid(True)
ax2.grid(False)

fig.savefig(os.path.join(output_folder, "kinetic_energy_2axes.png"))
plt.show()

In [ ]:
percentage_total_ke = 100 * eddy_ke / lake_ke[lake_ke.index.isin(eddy_ke.index)]

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
percentage_total_ke.plot(label='Entire lake')
plt.title('Fraction of total kinetic energy contained in eddies')
plt.ylabel('Fraction [%]')
plt.xlabel('')
plt.xticks(rotation=10)
plt.legend()
fig.savefig(os.path.join(output_folder, "fraction_ke.png"))

# Depth analysis

In [ ]:
df_lvl0['layer_thickness_[m]'] = df_lvl0['volume_slice_[m3]'] / df_lvl0['surface_area_[m2]']
df_lake['layer_thickness_[m]'] = df_lake['volume_slice_[m3]'] / df_lake['surface_area_[m2]']

In [ ]:
df_lvl0['ke_per_meter_[MJ/m]'] = df_lvl0['kinetic_energy_eddy_[MJ]'] / df_lvl0['layer_thickness_[m]']
df_lake['ke_per_meter_[MJ/m]'] = df_lake['kinetic_energy_[MJ]'] / df_lake['layer_thickness_[m]']

### Time-Depth plots

In [ ]:
profile_eddy_ke = df_lvl0.groupby(['date', 'depth_[m]', 'time_index'])[['ke_per_meter_[MJ/m]']].sum().reset_index()

In [ ]:
profile_eddy_ke.head()

In [ ]:
pivot_profile_eddy_ke = profile_eddy_ke.pivot(index="depth_[m]", columns="date", values="ke_per_meter_[MJ/m]")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_profile_eddy_ke.columns,
    pivot_profile_eddy_ke.index,
    pivot_profile_eddy_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Kinetic Energy contained in eddies")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Kinetic energy [MJ]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ke_eddy_time_depth.png"))
plt.show()

In [ ]:
pivot_lake_ke = df_lake.pivot(index="depth_[m]", columns="date", values="ke_per_meter_[MJ/m]")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_lake_ke.columns,
    pivot_lake_ke.index,
    pivot_lake_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Total Kinetic Energy")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Kinetic energy [MJ]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ke_total_time_depth.png"))
plt.show()

In [ ]:
pivot_ratio_ke = 100 * pivot_profile_eddy_ke / pivot_lake_ke

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_ratio_ke.columns,
    pivot_ratio_ke.index,
    pivot_ratio_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Ratio Kinetic Energy in eddies")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Ratio of Kinetic Energy [%]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ratio_ke_time_depth.png"))
plt.show()

### KE profiles

In [ ]:
str(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0]).replace(' ','_').replace(':','-')

In [ ]:
os.makedirs(os.path.join(output_folder, "profiles_ke"), exist_ok=True)
for t_idx in range(1,8760, 24):
    plt.close('all')
    fig, ax = plt.subplots(figsize=(5,7))

    # Lake KE profile
    df_lake[df_lake['time_index']==t_idx].plot(
        x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Lake KE'
    )

    # Eddy KE profile
    profile_eddy_ke[profile_eddy_ke['time_index']==t_idx].plot(
        x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Eddy KE'
    )

    ax.set_xlabel("Kinetic Energy [MJ/m]")
    ax.set_ylabel("Depth [m]")
    ax.legend()
    plt.title(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0])
    ax.set_xlim(left=0, right=280)
    #plt.show()
    str_time = str(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0]).replace(' ','_').replace(':','-')
    fig.savefig(os.path.join(output_folder, "profiles_ke", f"kinetic_energy_profile_{str_time}.png"))

In [ ]:
fig, ax = plt.subplots(figsize=(5,7))
profile_eddy_ke[profile_eddy_ke['time_index']==t_idx].plot(
    x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Eddy KE'
)
plt.title(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0])
plt.show()

# Specific depth analysis

In [ ]:
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

In [ ]:
i_depth_min = 16
i_depth_max = len(depths)-1

In [ ]:
def filter_by_depths(df, i_depth_min, i_depth_max):
    depth_filter = (
            (df['depth_index'] <= i_depth_max) &
            (df['depth_index'] >= i_depth_min)
        )
    return df[depth_filter]

In [ ]:
str_depth_min = str(round(depths.iloc[i_depth_min]['depth_[m]'], 2))
str_depth_max = str(round(depths.iloc[i_depth_max]['depth_[m]'], 2))

print(str_depth_min, str_depth_max)

In [ ]:
df_lvl0_filtered_by_depth = filter_by_depths(df_lvl0, i_depth_min, i_depth_max).set_index('time_index', drop=False)
df_lake_filtered_by_depth = filter_by_depths(df_lake, i_depth_min, i_depth_max).set_index('time_index', drop=False)

In [ ]:
lvl0_filtered_by_depth_sum = df_lvl0_filtered_by_depth.groupby('date').sum()
lake_filtered_by_depth_sum = df_lake_filtered_by_depth.groupby('date').sum()

In [ ]:
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].reset_index().to_csv(os.path.join(output_folder, f"ke_eddies_{str_depth_min}-{str_depth_max}m.csv"))
lake_filtered_by_depth_sum['kinetic_energy_[MJ]'].reset_index().to_csv(os.path.join(output_folder, f"ke_lake_{str_depth_min}-{str_depth_max}m.csv"))

In [ ]:
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'] = 100 * lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'] / lake_filtered_by_depth_sum['kinetic_energy_[MJ]']

In [ ]:
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'].plot(label=f'{str_depth_min}-{str_depth_max}m')

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
percentage_total_ke.plot(label='Entire lake')
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'].plot(label=f'{str_depth_min}-{str_depth_max}m')
plt.title('Fraction of total kinetic energy contained in eddies')
plt.ylabel('Fraction [%]')
plt.xlabel('')
plt.xticks(rotation=10)
plt.legend()
fig.savefig(os.path.join(output_folder, f"fraction_ke_{str_depth_min}-{str_depth_max}m.png"))

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
lake_filtered_by_depth_sum['kinetic_energy_[MJ]'].plot(label='Total')
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].plot(label='Eddies')
plt.legend()
plt.ylabel(f'Kinetic Energy {str_depth_min}-{str_depth_max}m [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
#ax.set_yscale('log')
fig.savefig(os.path.join(output_folder, f"kinetic_energy_{str_depth_min}-{str_depth_max}m.png"))

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
eddy_ke.plot(label='Entire depth')
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].plot(label=f'{str_depth_min}-{str_depth_max}m')
plt.legend()
plt.title('Kinetic Energy contained in eddies')
plt.ylabel(f'Kinetic Energy [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
fig.savefig(os.path.join(output_folder, f"eddy_kinetic_energy_{str_depth_min}-{str_depth_max}m.png"))

# Horizontal distribution

In [ ]:
import numpy as np

In [ ]:
def to_array_csvlike(x, dtype=float):
    if isinstance(x, str):
        return np.fromstring(x, sep=',', dtype=dtype)
    return np.asarray(x, dtype=dtype)

In [ ]:
cols = ['i_eddy_cells', 'j_eddy_cells']
df_lvl0[cols] = df_lvl0[cols].map(lambda x: to_array_csvlike(x, dtype=np.int32))

In [ ]:
import sys
sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
from utils_energy_analysis import *

In [ ]:
model = 'lucerne_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC']) + 1) * grid_resolution - grid_resolution / 2
ds['XC'] = np.arange(1, len(ds['XC']) + 1) * grid_resolution - grid_resolution / 2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

In [ ]:
mask = ds['THETA'].isel(time=0).values > 0

In [ ]:
aligned_u = ds.UVEL.rename({'XG':'XC'})
aligned_u['XC'] = ds['XC']

aligned_v = ds.VVEL.rename({'YG':'YC'})
aligned_v['YC'] = ds['YC']

aligned_w = ds.WVEL.rename({'Zl':'Z'})
aligned_w['Z'] = ds['Z']

In [ ]:
ke_tot = compute_ke(
    aligned_u,
    aligned_v,
    aligned_w,
    grid_resolution,
    grid_resolution,
    ds.drF)

In [ ]:
def as_int(x):
    if isinstance(x, np.ndarray):
        return x.astype(int)
    return int(x)

In [ ]:
ny = ke_tot.sizes["YC"]
nx = ke_tot.sizes["XC"]

EKE_map = np.zeros((ny, nx), dtype=np.float64)

for (t, z), grp in df_lvl0.groupby(["time_index", "depth_index"], sort=False):
    ke_slice = ke_tot.isel(time=int(t - 1), Z=int(z)).values  # (y, x)

    for row in grp.itertuples(index=False):
        j = as_int(row.j_eddy_cells)
        i = as_int(row.i_eddy_cells)
        EKE_map[j, i] += ke_slice[j, i]


In [ ]:
np.save(os.path.join(output_folder, "EKE_map.npy"), EKE_map)

In [ ]:
EKE_map[EKE_map==0] = np.nan

In [ ]:
plt.imshow(EKE_map, origin='lower')
plt.colorbar()